In [2]:
import numpy as np
from scipy.optimize import minimize
import pandas as pd

# Experimental quark masses (in GeV)
m_up_exp = np.array([0.0022, 1.27, 173.0])    # u, c, t
m_down_exp = np.array([0.0047, 0.095, 4.18])  # d, s, b
n_vals = np.array([0, 1, 2])

# Spiral model
def spiral_log_mass(params, n):
    lam, eps, omega, phi, logK = params
    return lam * n + eps * np.sin(omega * n + phi) + logK

# Objective: minimize squared log errors
def objective(params, n, log_m_exp):
    return np.sum((spiral_log_mass(params, n) - log_m_exp)**2)

# Initial guesses
init_up = [1.6, 0.4, 3.88, np.pi, np.log10(0.0005)]
init_down = [1.2, 0.3, 3.88, 4.2, np.log10(0.0005)]

# Log experimental masses
log_m_up_exp = np.log10(m_up_exp)
log_m_down_exp = np.log10(m_down_exp)

# Fit up-type
fit_up = minimize(objective, init_up, args=(n_vals, log_m_up_exp), method='Nelder-Mead')
params_up = fit_up.x

# Fit down-type
fit_down = minimize(objective, init_down, args=(n_vals, log_m_down_exp), method='Nelder-Mead')
params_down = fit_down.x

# Predict masses
def predict(params):
    return 10 ** spiral_log_mass(params, n_vals)

pred_up = predict(params_up)
pred_down = predict(params_down)

# Results
df_up = pd.DataFrame({
    'Quark': ['u', 'c', 't'],
    'n': n_vals,
    'Predicted Mass (GeV)': pred_up,
    'Experimental Mass (GeV)': m_up_exp,
    'Error (%)': 100 * (pred_up - m_up_exp) / m_up_exp
})

df_down = pd.DataFrame({
    'Quark': ['d', 's', 'b'],
    'n': n_vals,
    'Predicted Mass (GeV)': pred_down,
    'Experimental Mass (GeV)': m_down_exp,
    'Error (%)': 100 * (pred_down - m_down_exp) / m_down_exp
})

print("Fitted parameters (up-type):")
print(f"λ = {params_up[0]:.6f}, ε = {params_up[1]:.6f}, ω = {params_up[2]:.6f}, φ = {params_up[3]:.6f}, log₁₀(K) = {params_up[4]:.6f}")
print("\nFitted parameters (down-type):")
print(f"λ = {params_down[0]:.6f}, ε = {params_down[1]:.6f}, ω = {params_down[2]:.6f}, φ = {params_down[3]:.6f}, log₁₀(K) = {params_down[4]:.6f}")
print("\n--- Up-type fit ---")
print(df_up.to_string(index=False))
print("\n--- Down-type fit ---")
print(df_down.to_string(index=False))


Fitted parameters (up-type):
λ = 2.433780, ε = 0.219624, ω = 4.266376, φ = 3.658490, log₁₀(K) = -2.549045

Fitted parameters (down-type):
λ = 1.250780, ε = 0.276398, ω = 4.669596, φ = 5.381334, log₁₀(K) = -2.111075

--- Up-type fit ---
Quark  n  Predicted Mass (GeV)  Experimental Mass (GeV)  Error (%)
    u  0              0.002200                   0.0022  -0.000557
    c  1              1.270011                   1.2700   0.000875
    t  2            173.000556                 173.0000   0.000322

--- Down-type fit ---
Quark  n  Predicted Mass (GeV)  Experimental Mass (GeV)  Error (%)
    d  0              0.004700                   0.0047  -0.000106
    s  1              0.095001                   0.0950   0.000767
    b  2              4.179947                   4.1800  -0.001271
